# Data generation: parametric elliptic diffusion problem — Section 6.2

Generates all data for the PDE experiments of Section 6.2 (Figures 2–4, Table 1):
$-\nabla\cdot(a(x,s)\nabla y)=10$ on $\Omega=[0,1]^2$, $y|_{\partial\Omega}=0$, with the
**affine** coefficient
$a(x,s)=1.89+\sum_{p=1}^{P}x_p\,\sin(\pi p s_1)\,p^{-9/5}$,
parameters $x\sim U([-1,1]^P)$, $P_1$ Lagrange elements, solutions stored as vertex
values.

**Protocol** (differences to the original version are documented in the companion
report):
1. Primal weak form with $a$ itself — **no** `pow(...,-1)` inversion (that belongs to
   the mixed formulation of the original MLSPDE code).
2. Coefficient built from precompiled `Expression`s + `Constant`s: one FFC compile
   total, then ~ms per sample.
3. **10 independent training pools** per configuration (no overlapping subsamples).
4. Training meshes follow the theory coupling $h=\sqrt{\lambda_N}=e^{-0.5N^{1/3}}$,
   realised as an $n\times n$ mesh with $n=\lfloor e^{0.5N^{1/3}}\rfloor$, i.e.
   $n = 10,\,28,\,52,\,84,\,124$ for $N = 100,\,300,\,500,\,700,\,900$.
   (The paper quotes $h_{\mathrm{Train}}\approx 1/124$ for $N=900$ accordingly.)
5. Test reference: **1000** samples on a $320\times 320$ mesh (CG + AMG,
   rtol $10^{-12}$), which dominates the finest training resolution:
   $(h_{\mathrm{Test}}/h_{\mathrm{Train}})^2=(124/320)^2\approx 0.15$.
6. The test parameters are additionally solved **on each training mesh**
   (`y_test_coarse`) → pipeline floor (FEM + transfer error under perfect learning).
7. Deterministic seeds throughout.

Note on the meshes: the reference mesh is built with diagonal direction `"right"` and
the training meshes with `"left"`. This is immaterial for everything downstream — the
bilinear transfer in notebook 03 only uses the vertex coordinates of the tensor grid —
but it is the configuration under which the reported numbers were produced, so it is
kept unchanged.

Outputs: `final_test_320.npz`, `final320_train_N={N}.npz` for the $N$-experiment,
`pvar320_P={P}.npz` for the increasing-$P$ experiment.
Runtime: ~1.5 h for the $N$-experiment, ~2.5 h for the $P$-experiment (single CPU;
the 320² CG+AMG solves take ~1 s each).

## Verification of the constants quoted in the paper

The following cell needs neither FEniCS nor any solve; it reproduces the mesh
sizes and the $\delta$-admissibility constants stated in Section 6.2.

In [ ]:
# ---------------------------------------------------------------------------
# Cross-check of the constants quoted in Section 6.2. Pure numpy, runs in seconds,
# no PDE solves -- run this first if you only want to verify the paper's numbers.
# ---------------------------------------------------------------------------
import numpy as np

print("mesh coupling  h_Train = e^{-0.5 N^{1/3}}")
for N in [100, 300, 500, 700, 900]:
    n = int(np.exp(0.5 * N ** (1 / 3)))
    print(f"  N={N:4d}:  n={n:3d}  h_Train=1/{n}   (h_Test/h_Train)^2=(n/320)^2={(n/320)**2:.4f}")

print("\ndelta-admissibility of rho_p = 0.999 p^{-1/5}   (phi_0 = 1.89)")
crude10 = sum(p ** (-8 / 5) for p in range(1, 11)) / 0.999
print(f"  crude bound, P=10 : {crude10:.4f}   (paper: 1.8812), delta = {1.89 - crude10:.4f}")
crude_inf = sum(p ** (-8 / 5) for p in range(1, 200000)) / 0.999
print(f"  crude bound, P=inf: {crude_inf:.4f}  > 1.89, hence the sharp bound below")

t = np.linspace(0.0, 1.0, 2_000_001)                    # grid for the head
head = np.zeros_like(t)
for p in range(1, 51):
    head += p ** (-8 / 5) * np.abs(np.sin(np.pi * p * t))
head /= 0.999
L = sum(np.pi * p * p ** (-8 / 5) for p in range(1, 51)) / 0.999     # Lipschitz constant
head_sup = head.max() + L * (t[1] - t[0]) / 2           # certified supremum
tail = (5 / 3) * 50 ** (-3 / 5) / 0.999                 # integral comparison, p > 50
print(f"  sharp head sup <= {head_sup:.4f}   (paper: 1.6606; Lipschitz const {L:.1f})")
print(f"  tail           <= {tail:.4f}   (paper: 0.1596)")
print(f"  total          <= {head_sup + tail:.4f} = 1.89 - {1.89 - head_sup - tail:.3f}"
      f"   (paper: 1.8202 < 1.89 - 0.069)")

print("\nadmissible range of c_scale for c_p = c_scale * 0.999^2 p^{-2/5}")
print(f"  crude bound: c_scale > ({crude10:.4f}/1.89)^2 = {(crude10/1.89)**2:.4f}")
print(f"  sharp bound: c_scale > ({head_sup+tail:.4f}/1.89)^2 = {((head_sup+tail)/1.89)**2:.4f}")
print("  -> of the grid used in notebook 03, only c_scale = 0.99 is covered by the theory;")
print("     the smaller values are included to probe the sensitivity to this parameter.")

## Solver

In [ ]:
import time
import numpy as np
from dolfin import *

set_log_level(LogLevel.ERROR)
parameters["form_compiler"]["optimize"] = True

PI = 3.14159265359          # as in the original MLSPDE code


class EllipticSolver:
    """Parametric solver for -div(a grad y) = 10, y = 0 on the boundary, with
    a(x,s) = 1.89 + sum_p x_p sin(pi p s1) p^{-9/5}.

    The phi_p are compiled ONCE as Expressions, the parameters x_p are Constants;
    per sample only the Constants are reassigned and the system is re-solved."""

    def __init__(self, mesh, P, phi0=1.89, expo=1.8):
        self.mesh = mesh
        self.Hh = FunctionSpace(mesh, 'P', 1)                  # P1 Lagrange
        self.xs = [Constant(0.0) for _ in range(P)]
        phis = [Expression(f'sin({PI}*x[0]*{p})*pow({p},-{expo})', degree=1,
                           domain=mesh) for p in range(1, P + 1)]
        a = Constant(phi0)
        for c, ph in zip(self.xs, phis):
            a = a + c * ph                                     # affine structure
        y_tr, v = TrialFunction(self.Hh), TestFunction(self.Hh)
        self.bc = DirichletBC(self.Hh, Constant(0.0), lambda s, on_b: on_b)
        self.A = a * dot(grad(y_tr), grad(v)) * dx             # primal weak form
        self.L = Constant(10.0) * v * dx                       # f = 10
        self.y = Function(self.Hh)

    def solve(self, x, krylov=False):
        """Solve for parameter vector x; krylov=True -> CG+AMG (fine meshes)."""
        for c, xv in zip(self.xs, x):
            c.assign(float(xv))
        if krylov:
            sp = {"linear_solver": "cg", "preconditioner": "hypre_amg",
                  "krylov_solver": {"relative_tolerance": 1e-12,
                                    "absolute_tolerance": 1e-14}}
            solve(self.A == self.L, self.y, self.bc, solver_parameters=sp)
        else:
            solve(self.A == self.L, self.y, self.bc)           # direct solver
        return self.y.compute_vertex_values(self.mesh).copy()  # vertex values (P1)

In [ ]:
# ---------------- N-experiment (P=10): shared 1000-sample reference test set ----------------
P, N_TEST, K_REP = 10, 1000, 10

mesh_f = UnitSquareMesh(320, 320, "right")                     # dominant reference
sf = EllipticSolver(mesh_f, P)
rng = np.random.default_rng(31415)                             # test-set seed
x_test = rng.uniform(-1.0, 1.0, (N_TEST, P))
t0 = time.time()
y_test = np.array([sf.solve(x, krylov=True) for x in x_test], dtype=np.float32)
np.savez('final_test_320.npz', x_test=x_test, y_test=y_test,
         coords=mesh_f.coordinates())
print(f'fine test set: {time.time()-t0:.0f}s', flush=True)

for N in [100, 300, 500, 700, 900]:
    n = int(np.exp(0.5 * N ** (1 / 3)))                        # h = e^{-0.5 N^{1/3}}
    sc = EllipticSolver(UnitSquareMesh(n, n, "left"), P)
    t0 = time.time()
    x_pools, y_pools = [], []
    for r in range(K_REP):                                     # independent pools
        rng_r = np.random.default_rng(70000 + 100 * N + r)
        xp = rng_r.uniform(-1.0, 1.0, (N, P))
        x_pools.append(xp)
        y_pools.append(np.array([sc.solve(x) for x in xp], dtype=np.float32))
    y_test_coarse = np.array([sc.solve(x) for x in x_test], dtype=np.float32)
    np.savez(f'final320_train_N={N}.npz', x_pools=np.array(x_pools),
             y_pools=np.array(y_pools), coords=sc.mesh.coordinates(),
             y_test_coarse=y_test_coarse)
    print(f'N={N} (mesh {n}x{n}): {time.time()-t0:.0f}s', flush=True)

In [ ]:
# ---------------- P-experiment (N=300 fixed): per-P test sets on 320x320 ----------------
N, N_TEST, K_REP = 300, 1000, 10
n = int(np.exp(0.5 * N ** (1 / 3)))                            # = 28
mesh_c = UnitSquareMesh(n, n, "left")
mesh_f = UnitSquareMesh(320, 320, "right")

for P in [1, 10, 30, 50]:
    sf = EllipticSolver(mesh_f, P)
    sc = EllipticSolver(mesh_c, P)
    rng = np.random.default_rng(90000 + P)                     # test-set seed
    x_test = rng.uniform(-1.0, 1.0, (N_TEST, P))
    t0 = time.time()
    y_test = np.array([sf.solve(x, krylov=True) for x in x_test], dtype=np.float32)
    t1 = time.time()
    x_pools, y_pools = [], []
    for r in range(K_REP):                                     # independent pools
        rng_r = np.random.default_rng(91000 + 10 * P + r)
        xp = rng_r.uniform(-1.0, 1.0, (N, P))
        x_pools.append(xp)
        y_pools.append(np.array([sc.solve(x) for x in xp], dtype=np.float32))
    y_test_coarse = np.array([sc.solve(x) for x in x_test], dtype=np.float32)
    np.savez(f'pvar320_P={P}.npz', x_test=x_test, y_test=y_test,
             coords_test=mesh_f.coordinates(), coords_train=mesh_c.coordinates(),
             x_pools=np.array(x_pools), y_pools=np.array(y_pools),
             y_test_coarse=y_test_coarse)
    print(f'P={P}: fine test {t1-t0:.0f}s, pools+coarse {time.time()-t1:.0f}s', flush=True)
print('ALL DONE', flush=True)